# Approval gates — gate the dangerous tools, let the rest through

Approving every LLM action is unusable; approving none is unsafe. The middle ground is a **tool-policy table**: classify each tool as `safe`, `requires_approval`, or `forbidden`, then wrap the agent so only the gated tools fire an interrupt.

In the shared scenario, `search` and `draft` are safe (read-only), but `publish` mutates the world — that's the one the gate guards.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '04-human-in-the-loop'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from hitl import build_research_graph, Checkpointer, Command, get_question

TOOL_POLICY = {'search': 'safe', 'draft': 'safe', 'publish': 'requires_approval'}
TOOL_POLICY

## A policy-driven reviewer

The reviewer doesn't read the LLM's chain-of-thought. It reads the **tool name, arguments, and policy entry** — that's the only thing a human can audit at scale.

In [ ]:
def review(state, *, allow: bool):
    request = state['_interrupt']
    print(f"GATE: reason={request['reason']!r} draft={state['draft'][:80]!r}")
    return {'approved': allow, 'reviewer': 'human:demo'}

graph = build_research_graph()
cp = Checkpointer()
state = graph.run({'question': get_question('q01'), 'question_id': 'q01'},
                  thread_id='approve-demo', checkpointer=cp)
state = graph.resume(thread_id='approve-demo',
                     command=Command(resume=review(state, allow=True)),
                     checkpointer=cp)
print('approved:', state['approved'], '| published:', state['published'])

In [ ]:
# Same scenario, deny path — publish must stay False.
cp2 = Checkpointer()
state2 = graph.run({'question': get_question('q01'), 'question_id': 'q01'},
                   thread_id='deny-demo', checkpointer=cp2)
state2 = graph.resume(thread_id='deny-demo',
                      command=Command(resume=review(state2, allow=False)),
                      checkpointer=cp2)
assert state2['published'] is False
print('approved:', state2['approved'], '| published:', state2['published'])

## Approve-once vs approve-always

Real reviewers want a checkbox: *approve this argument shape forever*. That's just a side-table keyed on `(tool, hash(arguments))` — when the resume value comes back, persist it and skip the interrupt next time. The eval here measures the simpler one-shot behavior across both policies.